# Water Quality Prediction: v2 Notebook 
Update:
- Clustering rows by regions using K-means
- GroupKFolds to prevent data leakage
- Random Forest -> LightGBM
- add 10 more features from terraclimate

** 클러스터 기반 결측치

In [40]:
# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

# Geospatial raster data handling with CRS support
import rioxarray as rxr

# Raster operations and spatial windowing
import rasterio
from rasterio.windows import Window

# Feature preprocessing and data splitting
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.spatial import cKDTree

from lightgbm import LGBMRegressor
import optuna
import lightgbm as lgb

# Machine Learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.cluster import KMeans
from sklearn.model_selection import GroupKFold
import numpy as np

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os 

In [3]:
Water_Quality_df=pd.read_csv('./data/water_quality_training_dataset.csv')
print(len(Water_Quality_df))
Water_Quality_df.head()

9319


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0


In [4]:
landsat_train_features = pd.read_csv('./data/landsat_features_training.csv')
print(len(landsat_train_features))
landsat_train_features.head()

9319


,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI
0,-28.760833,17.730278,02-01-2011,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595
1,-26.861111,28.884722,03-01-2011,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134
2,-26.450000,28.085833,03-01-2011,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805
3,-27.671111,27.236944,03-01-2011,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416
4,-27.356667,27.286389,03-01-2011,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683


In [5]:
Terraclimate_df = pd.read_csv('./data/terraclimate_features_training.csv')
print(len(Terraclimate_df))
Terraclimate_df.head()

9319


,Latitude,Longitude,Sample Date,pet,ppt,aet,def,pdsi,q,soil,tmax,tmin,vpd
0,-28.760833,17.730278,02-01-2011,174.2,32.7,31.100000,143.100000,3.65,1.6,0.0,36.100000,22.689999,2.92
1,-26.861111,28.884722,03-01-2011,124.1,51.1,53.800000,70.300000,0.66,2.6,12.8,27.160000,13.219999,0.95
2,-26.450000,28.085833,03-01-2011,127.5,62.7,60.800000,66.700005,-1.16,3.1,6.8,27.519999,14.090000,1.02
3,-27.671111,27.236944,03-01-2011,129.7,84.2,83.200005,46.500000,2.84,4.2,7.2,28.869999,14.639999,1.22
4,-27.356667,27.286389,03-01-2011,129.2,78.0,77.300000,51.900000,2.65,3.9,7.8,28.670000,14.690000,1.18


In [6]:
# Combine two datasets vertically (along columns) using pandas concat function.
def combine_two_datasets(dataset1,dataset2,dataset3):
    '''
    Returns a  vertically concatenated dataset.
    Attributes:
    dataset1 - Dataset 1 to be combined 
    dataset2 - Dataset 2 to be combined
    '''
    
    data = pd.concat([dataset1,dataset2,dataset3], axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

In [ ]:
# Combining ground data and final data into a single dataset.
wq_data = combine_two_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df)
    
# wq_data['Total Akalinity'] = np.log1p(wq_data['Total Alkalinity'])
# wq_data['Electrical Conductance'] = np.log1p(wq_data['Electrical Conductance'])
# wq_data['Dissolved Reactive Phosphorus'] = np.log1p(wq_data['Dissolved Reactive Phosphorus'])

print(wq_data.shape)
wq_data.head()

(9319, 22)


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,nir,green,swir16,swir22,...,pet,ppt,aet,def,pdsi,q,soil,tmax,tmin,vpd
0,-28.760833,17.730278,02-01-2011,128.912,6.320768,2.397895,11190.0,11426.0,7687.5,7645.0,...,174.2,32.7,31.100000,143.100000,3.65,1.6,0.0,36.100000,22.689999,2.92
1,-26.861111,28.884722,03-01-2011,74.720,5.099256,5.099866,17658.5,9550.0,13746.5,10574.0,...,124.1,51.1,53.800000,70.300000,0.66,2.6,12.8,27.160000,13.219999,0.95
2,-26.450000,28.085833,03-01-2011,89.254,6.352629,4.394449,15210.0,10720.0,17974.0,14201.0,...,127.5,62.7,60.800000,66.700005,-1.16,3.1,6.8,27.519999,14.090000,1.02
3,-27.671111,27.236944,03-01-2011,82.000,5.321057,4.624973,14887.0,10943.0,13522.0,11403.0,...,129.7,84.2,83.200005,46.500000,2.84,4.2,7.2,28.869999,14.639999,1.22
4,-27.356667,27.286389,03-01-2011,56.100,4.984291,5.023881,16828.5,9502.5,12665.5,9643.0,...,129.2,78.0,77.300000,51.900000,2.65,3.9,7.8,28.670000,14.690000,1.18


## Model Building

<p align="justify"> Now let us select the columns required for our model building exercise. We will consider only Band swir22, NDMI and MNDWI from the Landsat data and pet from Terraclimate dataset as our predictor variables. It does not make sense to use latitude and longitude as predictor variables, as they do not have any direct impact on predicting the water quality parameters.</p>


In [ ]:
def kmeans_regions(df, lat_col="Latitude", lon_col="Longitude", n_clusters=20, random_state=42):
    """
    example return: [1 1 1 0 0 0 2 2] when there are 3 clusters and 8 samples.
    """
    km = KMeans(n_clusters=n_clusters, random_state=random_state)
    clusters = km.fit_predict(df[[lat_col, lon_col]].to_numpy())
    return clusters

def region_splits(X, groups, n_splits=5):
    """
    Using GroupKFold, ensure that all samples from the same cluster are put together in either the training or validating set.

    n_splits = 5: train:val = 80:20

    example return:
    Fold1= train_idx:[0 1 2 6 7] + val_idx :[3 4 5]
    Fold2= train_idx:[3 4 5 6 7] + val_idx :[0 1 2]
    Fold3= train_idx:[0 1 2 3 4 5] + val_idx :[6 7]
    """
    gkf = GroupKFold(n_splits=n_splits)
    for train_idx, val_idx in gkf.split(X, groups=groups):
        yield train_idx, val_idx

def fit_median_imputer(X_train):
    # Changed: fit medians only on training fold
    return X_train.median(numeric_only=True)

def apply_median_imputer(X, medians):
    # Changed: apply training medians to any dataset
    return X.fillna(medians)

def impute_by_cluster(X, n_clusters=20, random_state=42):
    """
    클러스터별 중앙값으로 결측치를 채우고, 
    데이터프레임과 클러스터 정보를 반환합니다.
    """
    X_imputed = X.copy()
    
    # 1. 클러스터 생성 (이미 정의하신 kmeans_regions 함수 사용)
    clusters = kmeans_regions(X_imputed, lat_col="Latitude", lon_col="Longitude",
                               n_clusters=n_clusters, random_state=random_state)
    X_imputed['temp_cluster'] = clusters

    # 2. 결측치 컬럼 파악
    cols_with_nan = X_imputed.columns[X_imputed.isna().any()].tolist()
    
    if cols_with_nan:
        # 클러스터별 중앙값으로 채우기
        X_imputed[cols_with_nan] = X_imputed.groupby('temp_cluster')[cols_with_nan].transform(
            lambda x: x.fillna(x.median())
        )
        # 특정 클러스터 전체가 NaN일 경우를 대비해 전체 중앙값으로 최종 보간
        X_imputed[cols_with_nan] = X_imputed[cols_with_nan].fillna(X_imputed[cols_with_nan].median())
    
    # 3. 임시 컬럼 제거
    X_imputed = X_imputed.drop(columns=['temp_cluster'])
    
    return X_imputed, clusters

def feature_engineering(df):
    df = df.copy()  # Changed: avoid mutating original dataframe
    eps = 1e-10     # Changed

    dt = pd.to_datetime(df['Sample Date'], dayfirst=True, errors='coerce')
    df['month'] = dt.dt.month
    df['sin_month'] = np.sin(2 * np.pi * df['month'] / 12)
    df['cos_month'] = np.cos(2 * np.pi * df['month'] / 12)

    # Changed: remote sensing features
    df['NDVI'] = (df['nir'] - df['ls_red']) / (df['nir'] + df['ls_red'] + eps)
    df['NDBI'] = (df['swir16'] - df['nir']) / (df['swir16'] + df['nir'] + eps)

    df['nir_red_ratio'] = df['nir'] / (df['ls_red'] + eps)
    df['green_blue_ratio'] = df['green'] / (df['ls_blue'] + eps)
    df['swir1_swir2_ratio'] = df['swir16'] / (df['swir22'] + eps)

    df['nir_red_diff'] = df['nir'] - df['ls_red']
    df['green_swir1_diff'] = df['green'] - df['swir16']

    # Changed: climate interaction features
    df['ppt_pet_ratio'] = df['ppt'] / (df['pet'] + eps)
    df['aet_pet_ratio'] = df['aet'] / (df['pet'] + eps)
    df['water_surplus'] = df['ppt'] - df['pet']
    df['temp_range'] = df['tmax'] - df['tmin']
    df['soil_vpd'] = df['soil'] * df['vpd']

    # Changed: spatial interaction features
    df['lat2'] = df['Latitude'] ** 2
    df['lon2'] = df['Longitude'] ** 2
    df['lat_lon_interaction'] = df['Latitude'] * df['Longitude']

    df = df.drop(columns=['month', 'Sample Date'])

    return df

In [ ]:
def run_pipeline_optuna(X, y, param_name="Parameter", n_trials=80):

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    print(f"\n{'='*60}")
    print(f"Tuning Model for {param_name}")
    print(f"{'='*60}")

    X = X.copy()                     # Changed
    X = feature_engineering(X)       # Changed

    clusters = kmeans_regions(
        X,
        lat_col="Latitude",
        lon_col="Longitude",
        n_clusters=20,
        random_state=42
    )

    folds = list(region_splits(X, groups=clusters, n_splits=5))

    def objective(trial):

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 500, 2000),
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 16, 128),
            "max_depth": trial.suggest_int("max_depth", 4, 10),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 30),
            "subsample": trial.suggest_float("subsample", 0.75, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.75, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            "random_state": 42,
            "n_jobs": -1,
            "verbosity": -1,
        }

        fold_scores = []

        for fold, (train_idx, val_idx) in enumerate(folds):
            X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

            # Changed: fit imputer on train fold only
            medians = fit_median_imputer(X_train)
            X_train = apply_median_imputer(X_train, medians)
            X_val = apply_median_imputer(X_val, medians)

            model = lgb.LGBMRegressor(**params)

            model.fit(
                X_train,
                y_train,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(50, verbose=False)]
            )

            preds = model.predict(X_val)
            fold_scores.append(r2_score(y_val, preds))

        return float(np.mean(fold_scores))

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    print("\nBest R2:", study.best_value)
    print("Best Params:", study.best_params)

    best_params = study.best_params
    best_params.update({
        "verbosity": -1,
        "random_state": 42,
        "n_jobs": -1
    })

    # Changed: final preprocessing on full training data
    final_medians = fit_median_imputer(X)
    X_final = apply_median_imputer(X.copy(), final_medians)

    final_model = lgb.LGBMRegressor(**best_params)
    final_model.fit(X_final, y)

    return final_model, study.best_params, study.best_value, final_medians

In [ ]:
wq_data = wq_data.drop(columns=['swir16'])

X = wq_data.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])

print(X.columns.tolist())

['Latitude', 'Longitude', 'nir', 'green', 'swir22', 'NDMI', 'MNDWI', 'pet', 'ppt', 'aet', 'def', 'pdsi', 'q', 'soil', 'tmax', 'tmin', 'vpd', 'sin_month', 'cos_month']


In [15]:
y_TA = wq_data['Total Alkalinity']
y_EC = wq_data['Electrical Conductance']
y_DRP = wq_data['Dissolved Reactive Phosphorus']

model_TA, best_params_TA, r2_TA = run_pipeline_optuna(X, y_TA, "Total Alkalinity", n_trials=80)
model_EC, best_params_EC, r2_EC = run_pipeline_optuna(X, y_EC, "Electrical Conductance", n_trials=80)
model_DRP, best_params_DRP, r2_DRP = run_pipeline_optuna(X, y_DRP, "Dissolved Reactive Phosphorus", n_trials=80)


Tuning Model for Total Alkalinity

Best R2: 0.15816594954232296
Best Params: {'n_estimators': 1029, 'learning_rate': 0.021996929927734946, 'num_leaves': 21, 'max_depth': 9, 'min_child_samples': 26, 'subsample': 0.9601813976135134, 'colsample_bytree': 0.8055374625653557, 'reg_alpha': 0.000447435339864153, 'reg_lambda': 5.495441597486022}

Tuning Model for Electrical Conductance

Best R2: 0.19985357889833236
Best Params: {'n_estimators': 1294, 'learning_rate': 0.05778302389666278, 'num_leaves': 35, 'max_depth': 4, 'min_child_samples': 23, 'subsample': 0.8484271502342918, 'colsample_bytree': 0.907927158793951, 'reg_alpha': 1.8791047964710768, 'reg_lambda': 7.952925491987129}

Tuning Model for Dissolved Reactive Phosphorus

Best R2: -0.0069738660084788465
Best Params: {'n_estimators': 727, 'learning_rate': 0.032042630260960006, 'num_leaves': 57, 'max_depth': 4, 'min_child_samples': 29, 'subsample': 0.9049990974388481, 'colsample_bytree': 0.78423676321846, 'reg_alpha': 0.6718644955262052, 

In [16]:
print(X.isna().sum())
print(X.columns.tolist())
print(X.shape)

Latitude        0
Longitude       0
nir          1085
green        1085
swir22       1085
NDMI         1085
MNDWI        1085
pet             0
ppt             0
aet             0
def             0
pdsi            0
q               0
soil            0
tmax            0
tmin            0
vpd             0
sin_month       0
cos_month       0
dtype: int64
['Latitude', 'Longitude', 'nir', 'green', 'swir22', 'NDMI', 'MNDWI', 'pet', 'ppt', 'aet', 'def', 'pdsi', 'q', 'soil', 'tmax', 'tmin', 'vpd', 'sin_month', 'cos_month']
(9319, 19)


### Model Performance Summary

<p align="justify">After training and evaluating the models for each water quality parameter, the individual performance metrics are combined into a single summary table. This table consolidates the R² and RMSE values for both in-sample and out-of-sample evaluations, enabling an easy comparison of model performance across Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus. Such a summary provides a quick overview of how well each model captures the variability in the respective parameter and highlights any differences in predictive accuracy.</p>

In [17]:
results_summary = pd.DataFrame([  # Changed
    {"Parameter": "Total Alkalinity", "Val_R2": r2_TA, "Best_Params": best_params_TA},  # Changed
    {"Parameter": "Electrical Conductance", "Val_R2": r2_EC, "Best_Params": best_params_EC},  # Changed
    {"Parameter": "Dissolved Reactive Phosphorus", "Val_R2": r2_DRP, "Best_Params": best_params_DRP},  # Changed
])  # Changed

results_summary

,Parameter,Val_R2,Best_Params
0,Total Alkalinity,0.158166,"{'n_estimators': 1029, 'learning_rate': 0.0219..."
1,Electrical Conductance,0.199854,"{'n_estimators': 1294, 'learning_rate': 0.0577..."
2,Dissolved Reactive Phosphorus,-0.006974,"{'n_estimators': 727, 'learning_rate': 0.03204..."


## Submission

<p align="justify">Once you are satisfied with your model’s performance, you can proceed to make predictions for unseen data. To do this, use your trained model to estimate the concentrations of the target water quality parameters — Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus — for a set of test locations provided in the "Submission_template.csv" file. The predicted results can then be uploaded to the challenge platform for evaluation.</p>

In [30]:
#Reading the coordinates for the submission
test_file = pd.read_csv('./data/submission_template.csv')
test_file.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,NaN,NaN,NaN
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,NaN,NaN,NaN
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,NaN,NaN,NaN


<p align="justify">
Similarly, participants can use the <b>Landsat</b> and <b>TerraClimate</b> data extraction demonstration notebooks to produce feature CSVs for their <b>validation</b> data. For convenience, we have already computed and saved example validation outputs as <code>landsat_features_val_V3.csv</code> and <code>Terraclimate_val_df_v3.csv</code>. Participants should save their own extracted files in the same format and column schema; doing so will allow this benchmark notebook to load the validation features directly and run smoothly.
</p>


In [31]:
landsat_val_features = pd.read_csv('./data/landsat_features_validation.csv')
landsat_val_features.head()

,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI
0,-32.043333,27.822778,01-09-2014,15229.0,12868.0,14797.0,12421.0,0.014388,-0.069727
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,16221.0,9304.5,12536.5,9958.0,0.128123,-0.147979
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,9125.0,11100.5,9455.0,8711.0,-0.017761,0.080052


In [32]:
Terraclimate_val_df = pd.read_csv('./data/terraclimate_features_validation.csv')
Terraclimate_val_df.head()

,Latitude,Longitude,Sample Date,pet,ppt,aet,def,pdsi,q,soil,tmax,tmin,vpd
0,-32.043333,27.822778,01-09-2014,161.90001,25.6,24.800001,137.10000,-2.12,1.3,4.8,28.750000,15.929999,1.15
1,-33.329167,26.077500,16-09-2015,177.60000,11.6,11.200000,166.40001,-0.85,0.6,2.7,29.340000,15.339999,1.11
2,-32.991639,27.640028,07-05-2015,158.40001,21.5,22.100000,136.30000,-2.24,1.1,12.0,27.390000,17.869999,0.83
3,-34.096389,24.439167,07-02-2012,130.00000,28.2,34.100002,95.90000,2.85,1.4,16.4,23.769999,16.070000,0.44
4,-32.000556,28.581667,01-10-2014,152.50000,27.2,26.400000,126.10000,-2.34,1.4,5.1,27.800000,16.560000,0.89


In [33]:
# Changed: build val_data by merging on keys instead of relying on `.values` alignment
tc_keys = ['Latitude', 'Longitude', 'Sample Date']  # Changed

val_data = (
    landsat_val_features[
        ['Longitude','Latitude','Sample Date','nir','green','swir16','swir22','NDMI','MNDWI']
    ]  # Changed
    .merge(
        Terraclimate_val_df[
            tc_keys + ['pet','ppt','aet','def','pdsi','q','soil','tmax','tmin','vpd']  # Changed
        ],
        on=tc_keys,
        how='left',
        validate='one_to_one'  # Changed: catches duplicate/missing key issues
    )
)

feature_cols = X.columns.tolist()
val_data.drop(columns=['swir16'], inplace=True)
val_data = feature_engineering(val_data)

# imputation
val_data = imputate_by_median(val_data)
# val_data, _ = impute_by_cluster(val_data, n_clusters=20, random_state=42)
val_data = val_data.reindex(columns=feature_cols)
submission_val_data = val_data.copy()

# Changed: quick sanity checks (optional but useful)
print("val_data shape:", val_data.shape)  # Changed
print("val_data columns:", val_data.columns.tolist())  # Changed
# print("missing TerraClimate:", val_data[['pet','ppt','aet','def','pdsi','q','soil','tmax','tmin','vpd']].isna().mean().sort_values(ascending=False).head())  # Changed

val_data shape: (200, 19)
val_data columns: ['Latitude', 'Longitude', 'nir', 'green', 'swir22', 'NDMI', 'MNDWI', 'pet', 'ppt', 'aet', 'def', 'pdsi', 'q', 'soil', 'tmax', 'tmin', 'vpd', 'sin_month', 'cos_month']


In [34]:
print(submission_val_data.isna().sum())

Latitude     0
Longitude    0
nir          0
green        0
swir22       0
NDMI         0
MNDWI        0
pet          0
ppt          0
aet          0
def          0
pdsi         0
q            0
soil         0
tmax         0
tmin         0
vpd          0
sin_month    0
cos_month    0
dtype: int64


In [35]:
print("train features:", len(feature_cols), feature_cols[:5], "...")  # optional
print("submission features:", submission_val_data.shape[1])           # optional

train features: 19 ['Latitude', 'Longitude', 'nir', 'green', 'swir22'] ...
submission features: 19


In [ ]:
pred_TA_submission = model_TA.predict(submission_val_data)
pred_EC_submission = model_EC.predict(submission_val_data)
pred_DRP_submission = model_DRP.predict(submission_val_data)

# pred_TA_final = np.expm1(pred_TA_submission)
# pred_EC_final = np.expm1(pred_EC_submission)
# pred_DRP_final = np.expm1(pred_DRP_submission)

In [ ]:
submission_df = pd.DataFrame({
    'Longitude': test_file['Longitude'], 
    'Latitude': test_file['Latitude'],   
    'Sample Date': test_file['Sample Date'], 
    'Total Alkalinity': pred_TA_submission,
    'Electrical Conductance': pred_EC_submission,
    'Dissolved Reactive Phosphorus': pred_DRP_submission
})

In [38]:
#Displaying the sample submission dataframe
submission_df.head()

,Longitude,Latitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,27.822778,-32.043333,01-09-2014,68.853139,190.757746,17.870147
1,26.077500,-33.329167,16-09-2015,169.391993,691.982832,23.937665
2,27.640028,-32.991639,07-05-2015,62.773242,219.832664,20.834089
3,24.439167,-34.096389,07-02-2012,22.176865,247.224108,10.844395
4,28.581667,-32.000556,01-10-2014,52.516026,164.240150,16.988291


In [39]:
#Dumping the predictions into a csv file.
submission_df.to_csv("./submission/v3_logEC_DRP.csv",index = False)

### Upload submission file on platform

Upload the submission.csv on the <a href ="https://challenge.ey.com">platform</a> to get score generated on scoreboard.

## Conclusion

<div align ="justify">Now that you have learned a basic approach to model training, it’s time to try your own approach! Feel free to modify any of the functions presented in this notebook. We look forward to seeing your version of the model and the results. Best of luck with the challenge!</div>